In [1]:

from pathlib import Path
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = "data/allbirds.db"
CHART_DIR = Path("outputs/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)


def load_data():
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("""
            SELECT p.product_id, p.title, p.product_type, p.vendor,
                   p.tags, o.variant_id, o.variant_title, o.size,
                   o.price, o.available, o.scraped_at
            FROM products p
            JOIN observations o ON p.product_id = o.product_id
        """, conn)

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["available"] = pd.to_numeric(
        df["available"], errors="coerce"
    )
    df["scraped_at"] = pd.to_datetime(
        df["scraped_at"], errors="coerce", utc=True
    )

    return df.drop_duplicates(
        ["product_id", "variant_id", "scraped_at"]
    )


def save_chart(filename):
    plt.tight_layout()
    plt.savefig(CHART_DIR / filename, dpi=160, bbox_inches="tight")
    plt.close()


def main():
    df = load_data()
    if df.empty:
        raise SystemExit("No observation data available.")

    # 1. Price distribution
    prices = df["price"].dropna()
    if not prices.empty:
        plt.figure()
        plt.hist(prices, bins=20)
        plt.title("Distribution of Allbirds Variant Prices")
        plt.xlabel("Price (source currency)")
        plt.ylabel("Number of observations")
        save_chart("01_price_distribution.png")

    # 2. Average price by product type
    type_prices = df.dropna(subset=["price", "product_type"])
    if not type_prices.empty:
        plt.figure()
        type_prices.groupby("product_type")["price"].mean().sort_values().plot(
            kind="bar"
        )
        plt.title("Average Price by Product Type")
        plt.xlabel("Product type")
        plt.ylabel("Average price (source currency)")
        plt.xticks(rotation=45, ha="right")
        save_chart("02_average_price_by_type.png")

    # 3. Availability distribution
    avail = df["available"].map({
        1: "Available", 0: "Unavailable"
    }).fillna("Unknown").value_counts()

    plt.figure()
    avail.plot(kind="bar")
    plt.title("Variant Availability")
    plt.xlabel("Availability")
    plt.ylabel("Observation count")
    save_chart("03_availability.png")

    # 4. Daily average price
    daily_df = df.dropna(subset=["price", "scraped_at"]).copy()
    daily_df["date"] = daily_df["scraped_at"].dt.date

    if not daily_df.empty:
        plt.figure()
        daily_df.groupby("date")["price"].mean().plot(marker="o")
        plt.title("Daily Average Variant Price")
        plt.xlabel("Collection date")
        plt.ylabel("Average price (source currency)")
        plt.xticks(rotation=45, ha="right")
        save_chart("04_daily_average_price.png")

    # 5. Daily available variant observations
    available_df = df.dropna(subset=["scraped_at"]).copy()
    available_df["date"] = available_df["scraped_at"].dt.date
    available_df["is_available"] = (
        available_df["available"] == 1
    ).astype(int)

    plt.figure()
    available_df.groupby("date")["is_available"].sum().plot(marker="o")
    plt.title("Daily Available Variant Observations")
    plt.xlabel("Collection date")
    plt.ylabel("Available variant observations")
    plt.xticks(rotation=45, ha="right")
    save_chart("05_daily_available_variants.png")

    # 6. Price versus availability
    box_data = []
    labels = []

    for value, label in [(0, "Unavailable"), (1, "Available")]:
        subset = df.loc[df["available"] == value, "price"].dropna()
        if not subset.empty:
            box_data.append(subset)
            labels.append(label)

    if box_data:
        plt.figure()
        plt.boxplot(box_data, tick_labels=labels)
        plt.title("Price Distribution by Availability")
        plt.xlabel("Availability")
        plt.ylabel("Price (source currency)")
        save_chart("06_price_by_availability.png")

    print("Charts saved to:", CHART_DIR.resolve())
    print("Rows analysed:", len(df))
    print("Distinct collection dates:", df["scraped_at"].dt.date.nunique())


if __name__ == "__main__":
    main()


Charts saved to: D:\Desktop\WEB SCRAPING\outputs\charts
Rows analysed: 52178
Distinct collection dates: 5
